In [20]:
import pandas as pd
import spacy
import medspacy
from medspacy.ner import TargetRule

# --- Setup medspacy with TargetMatcher (run once) ---
nlp = medspacy.load()
target_matcher = nlp.get_pipe("medspacy_target_matcher")

# Define cancer-related terms to detect
cancer_rules = [
    TargetRule("cancer", "PROBLEM"),
    TargetRule("cancers", "PROBLEM"),
    TargetRule("tumor", "PROBLEM"),
    TargetRule("tumour", "PROBLEM"),
    TargetRule("tumors", "PROBLEM"),
    TargetRule("tumours", "PROBLEM"),
    TargetRule("malignant", "PROBLEM"),
    TargetRule("malignancy", "PROBLEM"),
    TargetRule("carcinoma", "PROBLEM"),
    TargetRule("adenocarcinoma", "PROBLEM"),
    TargetRule("sarcoma", "PROBLEM"),
    TargetRule("melanoma", "PROBLEM"),
    TargetRule("glioma", "PROBLEM"),
    TargetRule("glioblastoma", "PROBLEM"),
    TargetRule("leukemia", "PROBLEM"),
    TargetRule("leukaemia", "PROBLEM"),
    TargetRule("lymphoma", "PROBLEM"),
    TargetRule("myeloma", "PROBLEM"),
    TargetRule("metastatic", "PROBLEM"),
    TargetRule("metastasis", "PROBLEM"),
    TargetRule("metastases", "PROBLEM"),
    TargetRule("neoplasm", "PROBLEM"),
    TargetRule("neoplastic", "PROBLEM"),
]

target_matcher.add(cancer_rules)
print(f"✓ Loaded medspacy with {len(cancer_rules)} cancer target rules")


def analyze_text_with_medspacy(text: str) -> dict:
    """
    Analyze text using medspacy with TargetMatcher and context detection.
    Returns counts of affirmed vs negated cancer mentions.
    """
    if not text or not isinstance(text, str) or not text.strip():
        return {
            "affirmed_count": 0,
            "negated_count": 0,
            "uncertain_count": 0
        }

    doc = nlp(text.lower())

    affirmed = 0
    negated = 0
    uncertain = 0

    for ent in doc.ents:
        if ent.label_ == "PROBLEM":  # Our cancer entities
            if hasattr(ent._, "is_negated") and ent._.is_negated:
                negated += 1
            elif hasattr(ent._, "is_uncertain") and ent._.is_uncertain:
                uncertain += 1
            elif hasattr(ent._, "is_hypothetical") and ent._.is_hypothetical:
                uncertain += 1
            else:
                affirmed += 1

    return {
        "affirmed_count": affirmed,
        "negated_count": negated,
        "uncertain_count": uncertain
    }


def classify_cancer_samples(df: pd.DataFrame) -> pd.DataFrame:
    """
    Classify samples as cancer / non-cancer / uncertain using medspacy.
    Returns the same DataFrame with added classification columns.
    """
    
    # Make a copy to avoid modifying the original
    df = df.copy()

    # --- Step 1: Setup ---
    PRIORITY_COLS = ["source_name", "tissue", "phenotype", "disease", "cell_type", "tumor_type"]
    PRIORITY_COLS = [c for c in PRIORITY_COLS if c in df.columns]

    # Onco-traps pattern (still useful for false positives)
    ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"

    # Helper to normalize text columns
    def normalize_text(series):
        return (
            series.astype(str)
            .fillna("")
            .str.lower()
            .str.replace(r"[_/|]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    # --- Step 2: Sample name detection using medspacy ---
    sample_name_col = "title" if "title" in df.columns else "biosample"

    # Extract texts for sample names
    sample_texts = normalize_text(df[sample_name_col]).tolist()

    # Analyze each sample name with medspacy
    sample_results = [analyze_text_with_medspacy(text) for text in sample_texts]

    df["cancer_in_sample_name"] = [r["affirmed_count"] > 0 for r in sample_results]
    df["negative_in_sample_name"] = [r["negated_count"] > 0 for r in sample_results]
    df["sample_name_affirmed_count"] = [r["affirmed_count"] for r in sample_results]
    df["sample_name_negated_count"] = [r["negated_count"] for r in sample_results]
    df["onco_trap_in_sample_name"] = normalize_text(df[sample_name_col]).str.contains(ONCO_TRAPS, regex=True)

    # --- Step 3: Check priority columns with medspacy ---
    for col in PRIORITY_COLS:
        col_texts = normalize_text(df[col]).tolist()
        col_results = [analyze_text_with_medspacy(text) for text in col_texts]

        df[f"cancer_in_{col}"] = [r["affirmed_count"] > 0 for r in col_results]
        df[f"negative_in_{col}"] = [r["negated_count"] > 0 for r in col_results]
        df[f"{col}_affirmed_count"] = [r["affirmed_count"] for r in col_results]
        df[f"{col}_negated_count"] = [r["negated_count"] for r in col_results]

    # --- Step 4: Count mentions across all priority columns ---
    cancer_mention_cols = [f"cancer_in_{c}" for c in PRIORITY_COLS]
    negative_mention_cols = [f"negative_in_{c}" for c in PRIORITY_COLS]

    # Count total affirmed cancer mentions (weighted by counts)
    affirmed_count_cols = ["sample_name_affirmed_count"] + [f"{c}_affirmed_count" for c in PRIORITY_COLS]
    negated_count_cols = ["sample_name_negated_count"] + [f"{c}_negated_count" for c in PRIORITY_COLS]

    df["n_cancer_mentions"] = df[[c for c in cancer_mention_cols if c in df.columns]].sum(axis=1)
    df["n_negative_mentions"] = df[[c for c in negative_mention_cols if c in df.columns]].sum(axis=1)
    df["total_affirmed_count"] = df[[c for c in affirmed_count_cols if c in df.columns]].sum(axis=1)
    df["total_negated_count"] = df[[c for c in negated_count_cols if c in df.columns]].sum(axis=1)

    # --- Step 5: Enhanced confidence category with negation awareness ---
    def assign_confidence(row):
        if row["onco_trap_in_sample_name"]:
            return "uncertain_onco_trap"
        
        # Strong negation signal - likely non-cancer
        if row["total_negated_count"] >= row["total_affirmed_count"] + 1:
            return "likely_non_cancer"
        
        # HIGH CONFIDENCE: Cancer in sample name AND in priority columns, no negations
        if (row["cancer_in_sample_name"] and 
            row["n_cancer_mentions"] >= 1 and 
            row["total_negated_count"] == 0):
            return "confident_cancer"
        
        # MEDIUM CONFIDENCE: Multiple cancer mentions, minimal negations
        if (((row["cancer_in_sample_name"] and row["total_negated_count"] == 0) or
             row["n_cancer_mentions"] >= 2) and
            row["total_negated_count"] <= row["total_affirmed_count"] // 2):
            return "likely_cancer"
        
        # LIKELY NON-CANCER: Negative indicators present
        if (row["negative_in_sample_name"] or 
            row["n_negative_mentions"] >= 1 or 
            row["total_negated_count"] > 0):
            return "likely_non_cancer"
        
        # UNCERTAIN: Weak signal
        if row["n_cancer_mentions"] == 1:
            return "uncertain_weak_signal"
        
        # UNCERTAIN: No cancer mentions at all
        return "uncertain_no_signal"

    df["confidence_category"] = df.apply(assign_confidence, axis=1)

    return df


print("✓ Classification function ready")


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject